<a href="https://colab.research.google.com/github/scottdzimmerman/GOGET/blob/main/goget_qc_clean_2026_11_10_Current.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GOGET QC Application

This notebook automates quality control checks in a user friendly manner. Click through the first cell, allow access on the second, add the db link to the secerets panel on the third, and select what kind of qc you want on the fourth cell.

In [ ]:
# Install Dependencies - No need to edit this
!pip install pandas sqlalchemy psycopg2-binary psycopg[binary] xlsxwriter shapely geopandas pyogrio ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 60.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ijson]


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
import os
import pytz
import json
import math
import unicodedata
import zipfile
import gc
import shutil
import tempfile
from typing import List
from google.colab import files
from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')


# ============================================================================
# COORDINATE CHECK — land (Natural Earth 50m) + maritime (Marine Regions EEZ)
# ============================================================================
# Replaces the old "nearest land polygon" logic, which mis-assigned every
# offshore field to whatever coastline happened to be closest (South China Sea
# -> Hong Kong, Danish North Sea -> Netherlands, NW Shelf -> Ashmore & Cartier).
#
# Logic per point:
#   1. Inside a land polygon?      -> onshore,  candidates = that country's names
#   2. Inside an EEZ polygon?      -> offshore, candidates = SOVEREIGN1-3 + TERRITORY1-3
#   3. Within SNAP_KM of an EEZ?   -> offshore (snapped), same candidates
#   4. Otherwise                   -> high seas, report nearest EEZ
# A point passes if the GOGET country matches ANY candidate. Overlapping-claim
# and joint-regime polygons carry 2-3 sovereigns, so disputed water passes for
# either claimant. If it fails, we measure the distance to the claimed country
# (land or EEZ, whichever is nearer) and grade by that distance.

os.environ.setdefault('OGR_GEOJSON_MAX_OBJ_SIZE', '0')  # EEZ features exceed GDAL's default cap

try:
    import geopandas as gpd
    from shapely.geometry import Point
    from shapely.ops import nearest_points
    from shapely.strtree import STRtree
    GEO_AVAILABLE = True
except ImportError:
    GEO_AVAILABLE = False

# Natural Earth carries several name variants per country. Reading only NAME
# (the short cartographic label) is why "S. Sudan", "Eq. Guinea" and
# "Ashmore and Cartier Is." were flagged. SOVEREIGNT also folds dependencies
# into their parent state, which resolves Hong Kong -> China.
LAND_NAME_FIELDS = ['NAME', 'NAME_LONG', 'ADMIN', 'SOVEREIGNT', 'GEOUNIT', 'SUBUNIT', 'BRK_NAME', 'NAME_CIAWF']
EEZ_NAME_FIELDS = ['SOVEREIGN1', 'SOVEREIGN2', 'SOVEREIGN3', 'TERRITORY1', 'TERRITORY2', 'TERRITORY3']

ALIAS_GROUPS = [
    {"united states", "united states of america", "usa", "us", "u.s.a."},
    {"russia", "russian federation"},
    {"iran", "iran (islamic republic of)", "islamic republic of iran"},
    {"south korea", "republic of korea", "korea, republic of", "korea rep."},
    {"north korea", "democratic peoples republic of korea", "dem. rep. korea"},
    {"bolivia", "bolivia (plurinational state of)"},
    {"venezuela", "venezuela (bolivarian republic of)"},
    {"taiwan", "taiwan, province of china", "chinese taipei"},
    {"united kingdom", "uk", "great britain", "britain"},
    {"democratic republic of the congo", "dr congo", "dem. rep. congo", "congo, dem. rep.", "congo-kinshasa", "zaire"},
    {"republic of the congo", "congo", "congo-brazzaville", "congo, rep."},
    {"ivory coast", "cote divoire", "cote d ivoire"},
    {"czech republic", "czechia"},
    {"myanmar", "burma"},
    {"laos", "lao pdr", "lao peoples democratic republic"},
    {"vietnam", "viet nam"},
    {"syria", "syrian arab republic"},
    {"tanzania", "united republic of tanzania"},
    {"moldova", "republic of moldova"},
    {"turkey", "turkiye", "republic of turkiye"},
    {"palestine", "state of palestine", "west bank", "palestinian territory", "gaza"},
    {"east timor", "timor-leste", "timor leste"},
    {"cape verde", "cabo verde"},
    {"swaziland", "eswatini"},
    {"macedonia", "north macedonia"},
    {"brunei", "brunei darussalam"},
    {"mauritius", "republic of mauritius"},
    {"comoros", "comores"},
    {"netherlands", "kingdom of the netherlands", "holland"},
    {"falkland islands", "falkland is.", "malvinas", "falkland islands (malvinas)",
     "falkland / malvinas islands", "falkland malvinas islands"},
    {"faroe islands", "faeroe islands", "faroe is."},
    {"wake island", "wake island / enenkio", "wake island enenkio"},
]

_ABBREV = {"st.": "saint", "s.": "south", "n.": "north", "e.": "east", "w.": "west",
           "is.": "islands", "i.": "island", "eq.": "equatorial", "rep.": "republic",
           "dem.": "democratic", "fr.": "french", "sar": ""}


def _norm(s):
    """Casefold, strip diacritics (Türkiye -> turkiye), expand NE abbreviations."""
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c)).lower().strip()
    for junk in [" s.a.r.", " sar", " (the)"]:
        s = s.replace(junk, "")
    toks = [_ABBREV.get(t, t) for t in s.replace("'", "").replace("\u2019", "").split()]
    s = " ".join(t for t in toks if t)
    return " ".join(s.replace("-", " ").replace(",", " ").replace(".", " ").split())


_ALIAS_LOOKUP = {}
for _i, _g in enumerate(ALIAS_GROUPS):
    for _m in _g:
        _ALIAS_LOOKUP.setdefault(_norm(_m), set()).add(_i)


def _names_match(a, b):
    """Exact-or-alias only. The old substring fallback silently passed
    Niger/Nigeria, Sudan/South Sudan, Guinea/Equatorial Guinea, so it is gone."""
    na, nb = _norm(a), _norm(b)
    if not na or not nb:
        return False
    if na == nb:
        return True
    return bool(_ALIAS_LOOKUP.get(na, set()) & _ALIAS_LOOKUP.get(nb, set()))


def _haversine_km(p1, p2):
    lon1, lat1, lon2, lat2 = map(math.radians, [p1.x, p1.y, p2.x, p2.y])
    a = (math.sin((lat2 - lat1) / 2) ** 2
         + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2)
    return 6371.0088 * 2 * math.asin(math.sqrt(a))



# ---------------------------------------------------------------------------
# Contested or undelimited maritime boundaries.
#
# The EEZ layer draws a single hard line everywhere. Where Marine Regions has
# an explicit "Overlapping claim" or "Joint regime" polygon (Falklands, Western
# Sahara, Gibraltar, Nigeria-Sao Tome) that polygon carries every claimant and
# the check already passes for any of them. But Marine Regions has NO such
# polygon for several boundaries that are in fact undelimited -- it simply
# picks a line. Verified for DRC/Angola: the layer holds three plain 200NM
# polygons (Angola 495,866 km2; DRC 13,373 km2; Rep. of Congo 33,807 km2), no
# Cabinda polygon and no ZIC polygon, so points in the Zone of Common Interest
# resolve to Angola with no indication the boundary is contested.
#
# Listing a pair here does NOT suppress the row. It re-labels the verdict as
# 'disputed-boundary' so the reviewer sees that the EEZ layer's answer is not
# authoritative, rather than being told GOGET is wrong and nudged to "correct"
# the country to whichever side Marine Regions happened to pick.
DISPUTED_PAIRS = {
    frozenset({"democratic republic of the congo", "angola"}):
        "Undelimited. DRC contests the boundary and claims acreage in Angolan "
        "Blocks 14/15/17/31. Zone of Common Interest (ZIC) agreed 2007; Block "
        "14/23 PSC signed Dec 2023 (CABGOC operator); governance agreement "
        "operationalised through 2026. Marine Regions encodes the de facto "
        "Angolan line and has no ZIC or overlapping-claim polygon.",
    frozenset({"australia", "timor-leste"}):
        "2018 Timor Sea Maritime Boundary Treaty replaced the JPDA and moved "
        "the boundary south. Assignments predating it may be stale.",
    frozenset({"australia", "indonesia"}):
        "Timor Gap / seabed-vs-water-column boundaries differ; legacy acreage.",
    frozenset({"thailand", "malaysia"}):
        "Malaysia-Thailand Joint Development Area (MTJDA); Marine Regions "
        "assigns the whole JDA to Malaysia.",
    frozenset({"kuwait", "saudi arabia"}):
        "Partitioned (Divided) Zone, onshore and offshore, jointly operated.",
    frozenset({"senegal", "mauritania"}):
        "Grand Tortue Ahmeyim straddles the boundary under a 2018 ICA.",
    frozenset({"malaysia", "vietnam"}):
        "PM-3 Commercial Arrangement Area.",
    frozenset({"venezuela", "trinidad and tobago"}):
        "Cross-border unitisation (Loran-Manatee, Manakin-Cocuina).",
    frozenset({"angola", "republic of the congo"}):
        "Lianzi unitisation and the Angola-Congo JDZ north of the ZIC.",
    frozenset({"iran", "saudi arabia"}):
        "Median line in the Farsi / Arabiyah-Hasbah area is not fully settled.",
    frozenset({"cameroon", "equatorial guinea"}):
        "Bioko median line; Etinde sits close to it.",
    frozenset({"mozambique", "france"}):
        "Juan de Nova / Iles Eparses claimed by France, Madagascar and "
        "contested in Mozambique Channel licensing.",
}


def _canon(s):
    """Collapse a country name to a single canonical token via the alias table,
    so 'East Timor' (Marine Regions) and 'Timor-Leste' (GOGET) resolve alike."""
    n = _norm(s)
    g = _ALIAS_LOOKUP.get(n)
    return f"__alias{min(g)}" if g else n


# Rebuild the pair keys in canonical space, once, at import.
DISPUTED_PAIRS = {frozenset(_canon(x) for x in k): v for k, v in DISPUTED_PAIRS.items()}


def _disputed_note(claim, candidates):
    """Return the dispute note if the claimed country and the containing
    jurisdiction are a known contested pair, else None."""
    a = _canon(claim)
    for c in candidates:
        note = DISPUTED_PAIRS.get(frozenset({a, _canon(c)}))
        if note:
            return note
    return None


# Fields that legitimately sit across a maritime boundary or in a joint
# development area. They are reported as "expected-cross-border" rather than
# MISMATCH so they stop consuming review time every cycle. Keyed by project_id.
EXPECTED_CROSS_BORDER = {
    # Malaysia-Thailand Joint Development Area (MTJDA) -- MR assigns to Malaysia
    "CHESS/Block A-18", "CPOC", "Bumi", "Bulan", "Muda", "Suriya", "Cakerawala",
    # Saudi-Kuwait Partitioned/Divided Zone
    "Khafji", "Lulu", "Onshore Neutral Zone",
    # Senegal-Mauritania straddling field
    "Ahmeyim",
    # Malaysia-Vietnam Commercial Arrangement Area (PM-3 CAA)
    "Bunga Lavatera", "Bunga Aster",
    # Venezuela-Trinidad cross-border unitisation
    "Manakin-Cocuina", "Cocuina", "Loran-Manatee", "Frangipani",
    # Angola / Congo-Brazzaville unitisation
    "Lianzi",
}

_GEO_CACHE = {}
_EEZ_META = {}
EEZ_LAYER = 'eez'
EEZ_COLS = ['MRGID', 'GEONAME', 'POL_TYPE'] + EEZ_NAME_FIELDS


# ---------------------------------------------------------------------------
# EEZ storage. A GeoJSON cannot be read partially: GDAL parses the WHOLE file into RAM (roughly
# 3-4x its size) before applying any bbox, which is what crashed Colab. So the GeoJSON is streamed
# once, one feature at a time, into a GeoPackage. A GeoPackage has a spatial index and can be read
# a few polygons at a time.
# ---------------------------------------------------------------------------
def build_eez_gpkg(geojson_path, gpkg_path, batch=8):
    try:
        import ijson
    except ImportError:
        raise RuntimeError("The one-off EEZ conversion needs ijson. Run  !pip install ijson  and try again.")
    from shapely.geometry import shape
    import pyogrio
    tmp = os.path.splitext(gpkg_path)[0] + '_partial.gpkg'
    if os.path.exists(tmp):
        os.remove(tmp)
    rows, geoms, seen, written = [], [], 0, 0

    def flush():
        nonlocal rows, geoms, written
        if not rows:
            return
        df = pd.DataFrame(rows, columns=EEZ_COLS)
        df['MRGID'] = pd.to_numeric(df['MRGID'], errors='coerce').astype('Int64')
        pyogrio.write_dataframe(gpd.GeoDataFrame(df, geometry=geoms, crs='EPSG:4326'), tmp,
                                layer=EEZ_LAYER, driver='GPKG', append=True, promote_to_multi=True)
        written += len(rows)
        rows, geoms = [], []

    with open(geojson_path, 'rb') as f:
        for feat in ijson.items(f, 'features.item', use_float=True):
            seen += 1
            if not feat.get('geometry'):
                continue
            props = {str(k).upper(): v for k, v in (feat.get('properties') or {}).items()}
            rows.append([props.get(c) for c in EEZ_COLS])
            geoms.append(shape(feat['geometry']))
            if len(rows) >= batch:
                flush()
            if seen % 50 == 0:
                print(f"    ...{seen} EEZ features converted")
    flush()
    os.replace(tmp, gpkg_path)
    return written


def _ensure_eez_gpkg(eez_path):
    """Return a local GeoPackage of the EEZ, building it once if it does not exist yet.
    The GeoPackage is kept next to the GeoJSON so later sessions just copy it (seconds, not minutes)."""
    if str(eez_path).lower().endswith('.gpkg'):
        return eez_path
    base = os.path.splitext(os.path.basename(eez_path))[0].replace(' ', '_') + '.gpkg'
    local_dir = '/content' if os.path.isdir('/content') else tempfile.gettempdir()
    local = os.path.join(local_dir, base)
    shared = os.path.splitext(eez_path)[0] + '.gpkg'
    src_time = os.path.getmtime(eez_path)

    def fresh(p):
        return os.path.exists(p) and os.path.getmtime(p) >= src_time

    if fresh(local):
        return local
    if fresh(shared):
        print("  Copying EEZ GeoPackage from the shared folder...")
        shutil.copy(shared, local)
        return local
    print("  One-time EEZ conversion (streams the GeoJSON with low memory; a few minutes)...")
    n = build_eez_gpkg(eez_path, local)
    print(f"  Converted {n} EEZ polygons.")
    try:
        shutil.copy(local, shared)
        print(f"  Saved a copy next to the GeoJSON so future sessions skip this step: {os.path.basename(shared)}")
    except Exception as e:
        print(f"  (Could not save a copy to the shared folder: {e}. It will be rebuilt next session.)")
    return local


def _eez_meta(gpkg):
    """Feature ids, bounding boxes and country names for every EEZ polygon. Tiny: no geometry is read."""
    if gpkg not in _EEZ_META:
        import pyogrio
        fids, bounds = pyogrio.read_bounds(gpkg, layer=EEZ_LAYER)
        meta = pyogrio.read_dataframe(gpkg, layer=EEZ_LAYER, columns=EEZ_NAME_FIELDS,
                                      read_geometry=False, fid_as_index=True)
        ecols = [c for c in EEZ_NAME_FIELDS if c in meta.columns]
        names = [[v for v in row if isinstance(v, str) and v]
                 for row in meta.loc[fids, ecols].itertuples(index=False)]
        _EEZ_META[gpkg] = (fids, bounds, names)
    return _EEZ_META[gpkg]


def _read_eez(gpkg, fids):
    import pyogrio
    if len(fids) == 0:
        return gpd.GeoDataFrame({c: [] for c in EEZ_COLS}, geometry=[], crs='EPSG:4326')
    return pyogrio.read_dataframe(gpkg, layer=EEZ_LAYER, fids=[int(i) for i in fids],
                                  columns=EEZ_COLS, fid_as_index=True)


class CountryResolver:
    """With `points` and `claim` given and a GeoPackage EEZ, only the EEZ polygons that can affect
    the answer for those points are read into memory:
      1. every polygon whose bounding box (padded by pad_deg) contains one of the points. A polygon
         that covers or is within snap distance of a point always has the point inside its padded
         box, so containment and snapping come out exactly as with the full file;
      2. every polygon belonging to the claimed country (wherever it is), so distance-to-claim is exact;
      3. for points that are neither on land nor inside a loaded polygon, the single nearest polygon
         in the WHOLE file, so the high-seas "nearest EEZ" is exact too.
    The verdicts are therefore the same as loading everything, using a fraction of the memory."""

    def __init__(self, land_path, eez_path=None, points=None, claim=None, pad_deg=0.5):
        lkey = ('land', land_path)
        if lkey not in _GEO_CACHE:
            print("  Loading land polygons...")
            land = gpd.read_file(land_path, engine='pyogrio')
            cols = [c for c in LAND_NAME_FIELDS if c in land.columns]
            lnames = [[v for v in row if isinstance(v, str) and v and v != '-99']
                      for row in land[cols].itertuples(index=False)]
            _GEO_CACHE[lkey] = (land, lnames, STRtree(land.geometry.values))
        self.land, self.land_names, self.land_tree = _GEO_CACHE[lkey]

        self.eez, self.eez_names, self.eez_tree = None, [], None
        if eez_path and os.path.exists(eez_path):
            if str(eez_path).lower().endswith('.gpkg'):
                self.eez = self._load_eez_region(eez_path, points, claim, pad_deg)
            else:
                print("  Loading EEZ polygons (large file; this is the memory-hungry path)...")
                self.eez = gpd.read_file(eez_path, engine='pyogrio', columns=EEZ_COLS)
        if self.eez is not None:
            ecols = [c for c in EEZ_NAME_FIELDS if c in self.eez.columns]
            self.eez_names = [[v for v in row if isinstance(v, str) and v]
                              for row in self.eez[ecols].itertuples(index=False)]
            self.eez_tree = STRtree(self.eez.geometry.values)

        self.land_by_name, self.eez_by_name = {}, {}
        for i, names in enumerate(self.land_names):
            for n in names:
                self.land_by_name.setdefault(_norm(n), []).append(i)
        for i, names in enumerate(self.eez_names):
            for n in names:
                self.eez_by_name.setdefault(_norm(n), []).append(i)

    def _load_eez_region(self, gpkg, points, claim, pad_deg):
        import numpy as np
        import shapely
        fids, bounds, names = _eez_meta(gpkg)
        if points is None:
            return _read_eez(gpkg, fids)
        pts = np.asarray(points, dtype=float).reshape(-1, 2)          # lon, lat
        x, y = pts[:, 0], pts[:, 1]

        # 1 + 2: polygons near the points, plus every polygon of the claimed country
        want = np.zeros(len(fids), dtype=bool)
        for k in range(len(fids)):
            x0, y0, x1, y1 = bounds[:, k]
            want[k] = bool(((x >= x0 - pad_deg) & (x <= x1 + pad_deg)
                            & (y >= y0 - pad_deg) & (y <= y1 + pad_deg)).any())
        if claim is not None and _norm(claim):
            want |= np.array([any(_names_match(claim, nm) for nm in nms) for nms in names], dtype=bool)
        eez = _read_eez(gpkg, fids[want])

        # 3: points that are not on land and not inside any loaded polygon need the globally nearest polygon
        P = shapely.points(x, y)
        on_land = set(self.land_tree.query(P, predicate='covered_by')[0].tolist())
        in_eez = set()
        tree = STRtree(eez.geometry.values) if len(eez) else None
        if tree is not None:
            in_eez = set(tree.query(P, predicate='covered_by')[0].tolist())
        loaded = set(int(f) for f in eez.index)
        extra, gcache = [], {}
        for i in range(len(P)):
            if i in on_land or i in in_eez:
                continue
            pt = P[i]
            best = float('inf')
            if tree is not None:
                j = tree.query_nearest(pt, all_matches=False)
                if len(j):
                    best = pt.distance(eez.geometry.values[int(j[0])])
            dx = np.maximum(np.maximum(bounds[0] - x[i], 0.0), x[i] - bounds[2])
            dy = np.maximum(np.maximum(bounds[1] - y[i], 0.0), y[i] - bounds[3])
            lb = np.hypot(dx, dy)                  # planar distance to each polygon's bbox: a lower bound
            order = np.argsort(lb)
            for k in order:
                f = int(fids[k])
                if lb[k] > best:
                    break
                if f in loaded:
                    continue
                if f not in gcache:
                    if len(gcache) >= 6:
                        gcache.pop(next(iter(gcache)))
                    gcache[f] = _read_eez(gpkg, [f]).geometry.values[0]
                d = pt.distance(gcache[f])
                if d <= best:                      # ties kept, like the full-file nearest query
                    best = d
                    if f not in extra:
                        extra.append(f)
        if extra:
            eez = pd.concat([eez, _read_eez(gpkg, extra)])
        eez = eez.sort_index()     # file order, so ties between overlapping polygons resolve the same way every time
        print(f"  EEZ polygons loaded for this region: {len(eez)} of {len(fids)}")
        return eez

    def _idx_for_claim(self, claim):
        n = _norm(claim)
        li, ei = set(self.land_by_name.get(n, [])), set(self.eez_by_name.get(n, []))
        if not li and not ei:
            groups = _ALIAS_LOOKUP.get(n, set())
            if groups:
                for k, v in self.land_by_name.items():
                    if _ALIAS_LOOKUP.get(k, set()) & groups:
                        li.update(v)
                for k, v in self.eez_by_name.items():
                    if _ALIAS_LOOKUP.get(k, set()) & groups:
                        ei.update(v)
        return sorted(li), sorted(ei)

    def _dist_km(self, pt, geoms):
        best = float('inf')
        for g in geoms:
            if g.covers(pt):
                return 0.0
            a, b = nearest_points(pt, g)
            best = min(best, _haversine_km(a, b))
        return best

    def resolve(self, lat, lon, claim, border_tol_km=5.0, snap_km=1.0):
        pt = Point(lon, lat)
        land_hits = [i for i in self.land_tree.query(pt)
                     if self.land.geometry.values[i].covers(pt)]
        eez_hits = []
        if self.eez_tree is not None and not land_hits:
            eez_hits = sorted(i for i in self.eez_tree.query(pt)
                              if self.eez.geometry.values[i].covers(pt))

        if land_hits:
            domain = 'onshore'
            cands = sorted({n for i in land_hits for n in self.land_names[i]})
            container = self.land_names[land_hits[0]][0]
        elif eez_hits:
            domain = 'offshore (EEZ)'
            cands = sorted({n for i in eez_hits for n in self.eez_names[i]})
            container = self.eez['GEONAME'].values[eez_hits[0]]
        elif self.eez_tree is not None:
            # Field coordinates are representative points; boundaries are median
            # lines. Snap anything sitting a few hundred metres outside.
            snap = sorted(i for i in self.eez_tree.query_nearest(pt, exclusive=False, all_matches=True)
                          if self._dist_km(pt, [self.eez.geometry.values[i]]) <= snap_km)
            if snap:
                domain = 'offshore (EEZ, snapped)'
                cands = sorted({n for i in snap for n in self.eez_names[i]})
                container = self.eez['GEONAME'].values[snap[0]]
            else:
                domain, cands, container, best = 'offshore (high seas)', [], None, float('inf')
                for i in self.eez_tree.query_nearest(pt, exclusive=False, all_matches=False):
                    d = self._dist_km(pt, [self.eez.geometry.values[i]])
                    if d < best:
                        best, container, cands = d, self.eez['GEONAME'].values[i], list(self.eez_names[i])
                container = f"{container} (nearest, {best:,.1f} km)"
        else:
            domain, cands, container = 'offshore (no EEZ layer)', [], None

        if any(_names_match(claim, c) for c in cands):
            return dict(domain=domain, container=container, verdict='OK',
                        dist_to_claim_km=0.0, candidates='; '.join(cands[:6]),
                        boundary_note='')

        # The containing polygon disagrees with GOGET. Before calling that an
        # error, check whether the boundary itself is contested.
        dispute = _disputed_note(claim, cands)

        li, ei = self._idx_for_claim(claim)
        d_land = self._dist_km(pt, [self.land.geometry.values[i] for i in li]) if li else float('inf')
        d_eez = self._dist_km(pt, [self.eez.geometry.values[i] for i in ei]) if ei else float('inf')
        d = min(d_land, d_eez)

        if d == 0:
            verdict = 'OK'
        elif d == float('inf'):
            verdict = 'unresolved-claim'   # GOGET country name not present in either layer
        elif dispute:
            verdict = 'disputed-boundary'  # EEZ layer's line is not authoritative here
        elif d <= border_tol_km:
            verdict = 'borderline'         # inside NE 50m generalisation error
        else:
            verdict = 'MISMATCH'
        return dict(domain=domain, container=container, verdict=verdict,
                    dist_to_claim_km=(None if d == float('inf') else round(d, 1)),
                    candidates='; '.join(cands[:6]),
                    boundary_note=(dispute or ''))


# ============================================================================
# DISCOVERY-YEAR ORDERING CHECK
# ============================================================================
# Error-level flag: a real discovery cannot be dated AFTER the unit is already
# in development or operating. Some "discovered" rows are really a source's
# report year (e.g. a registry "as of 31.XII.2022" mapped to discovered/2022),
# and this ordering test catches the subset that also has later milestones.
#
# Discovery year = earliest 'discovered' status_timeline row for the unit. A unit
# with no 'discovered' row has no discovery year and is never tested (it does not
# inherit an operating or any other status year).
#
# Exploration is deliberately NOT a milestone: exploration can legitimately come
# before discovery. 'first_exploration' is carried for information only.
# Same year is NOT a violation (strict >). Null years never flag a unit.
#
# KNOWN LIMITS
#  - Does not detect the "report-year" pattern when the field has no later
#    milestone (e.g. Polish PGI discovered/2022 fields with no FID/operating row).
#    That needs a separate source-level screen.
#  - A unit with no milestone after discovery cannot fail.
#  - Rows with a null year cannot be compared (count is printed in the run log).
#  - Year granularity only (monthOrHalfYear is ignored).
#  - Planned and actual entries are mixed in status_timeline. first_operating_any
#    includes planned start-ups. When the flattened/actual-only tables arrive,
#    edit DISCOVERY_ORDER_COMPARISONS (or pass cmp_cols) instead of the logic.
# This check only REPORTS. It never edits years; a human verifies against sources.

DISCOVERY_ORDER_COMPARISONS = [
    "first_actual_fid",            # in-development + substatus 'actual FID'
    "first_in_dev",                # in-development, any substatus
    "first_startup_nonplanned",    # operating, substatus != 'planned' (null substatus counts)
    "first_operating_any",         # operating, incl. planned
    "first_nondiscovered_row",     # any non-null status except discovered / exploration
]


def check_discovery_order(t: pd.DataFrame, cmp_cols: List[str] = None,
                          first_production: pd.Series = None) -> pd.DataFrame:
    """t = status_timeline rows (needs unit_id, status, substatus, year).
    Returns one row per unit whose earliest 'discovered' year is strictly later
    than any milestone year in cmp_cols, with a 'violated' column naming them.

    cmp_cols          which milestone columns are tested (default
                      DISCOVERY_ORDER_COMPARISONS).
    first_production  optional Series (index unit_id as str) of each unit's first
                      year with production > 0. If given it is carried as
                      'first_production_year' and tested as a separate, optional
                      comparison on top of cmp_cols; it never alters the five
                      status_timeline comparisons.
    """
    cmp_cols = list(cmp_cols) if cmp_cols else list(DISCOVERY_ORDER_COMPARISONS)
    unknown = [c for c in cmp_cols if c not in DISCOVERY_ORDER_COMPARISONS]
    if unknown:
        raise ValueError(f"Unknown comparison column(s) {unknown}; choose from {DISCOVERY_ORDER_COMPARISONS}")

    t = t[['unit_id', 'status', 'substatus', 'year']].copy()
    t['unit_id'] = t['unit_id'].astype(str)
    t['year'] = pd.to_numeric(t['year'], errors='coerce')

    disc = (t[t.status == "discovered"].groupby("unit_id").year.min().rename("disc_year"))

    def first_year(mask, name):
        return t[mask].groupby("unit_id").year.min().rename(name)

    other = t.status.notna() & ~t.status.isin(["discovered", "exploration"])
    parts = [
        disc,
        first_year((t.status == "in-development") & (t.substatus == "actual FID"), "first_actual_fid"),
        first_year(t.status == "in-development", "first_in_dev"),
        first_year((t.status == "operating") & (t.substatus != "planned"), "first_startup_nonplanned"),
        first_year(t.status == "operating", "first_operating_any"),
        first_year(other, "first_nondiscovered_row"),
        first_year(t.status == "exploration", "first_exploration"),   # informational only
    ]
    test_cols = list(cmp_cols)
    if first_production is not None:
        parts.append(pd.to_numeric(first_production, errors='coerce').rename("first_production_year"))
        test_cols.append("first_production_year")

    feat = pd.concat(parts, axis=1).dropna(subset=["disc_year"])

    bad = pd.Series(False, index=feat.index)
    for c in test_cols:
        bad |= feat.disc_year > feat[c]          # NaN compares False -> never flags
    out = feat[bad].copy()
    out["violated"] = [", ".join(c for c in test_cols if r.disc_year > r[c])
                       for _, r in out.iterrows()]
    out.index.name = "unit_id"
    return out.reset_index()


def run_qc_checks(database_url: str, output_dir: str, selected_countries: List[str] = None,
                  qc_steps: List[str] = None, geojson_path: str = None, eez_path: str = None,
                  outlier_ratio_threshold: int = 10, border_tol_km: float = 5.0,
                  flag_expected_cross_border: bool = True,
                  order_check_production: bool = False):
    if database_url.startswith('postgres:'):
        database_url = database_url.replace('postgres:', 'postgresql:')
    if 'cockroachlabs.cloud' in database_url:
        if '?verify-full=true' in database_url: database_url = database_url.replace('?verify-full=true', '?sslmode=require')
        elif '?sslmode=' not in database_url and '&sslmode=' not in database_url:
            database_url += '&sslmode=require' if '?' in database_url else '?sslmode=require'
    else:
        if '?verify-full=true' in database_url: database_url = database_url.replace('?verify-full=true', '?sslmode=require')

    os.makedirs(output_dir, exist_ok=True)
    now = datetime.now()
    timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")
    engine = create_engine(database_url)

    if not qc_steps: qc_steps = ["duplicates", "production", "reserves", "status", "discovery_order", "startups", "coordinates", "wkt"]

    # The land + EEZ polygons are cached between runs so repeat coordinate checks are fast,
    # but they hold a lot of RAM. If this run doesn't need them, let go of them.
    if "coordinates" not in qc_steps and _GEO_CACHE:
        _GEO_CACHE.clear()
        gc.collect()
        print("Released cached land/EEZ layers to free RAM.")

    def get_sql_table(table_name):
        return pd.read_sql(f"SELECT * FROM {table_name}", engine)

    def load_reserves_production(columns=None, project_ids=None, exclude_reserves=False):
        """Load reserves_production without pulling more than the selected checks need.
        columns          list of column names, or None for every column
        project_ids      only fetch rows for these projects (the later inner join would drop the rest anyway)
        exclude_reserves skip rows whose productionReserves is 'reserves' (IS DISTINCT FROM keeps NULLs,
                         matching the pandas `!= 'reserves'` test)
        The WHERE clauses are only an optimisation: if the filtered query fails for any reason we fall
        back to an unfiltered read and the pandas code below still produces the same result."""
        select = '*' if columns is None else ', '.join(f'"{c}"' for c in columns)
        base = f"SELECT {select} FROM reserves_production"
        where, params = [], {}
        if exclude_reserves:
            where.append('"productionReserves" IS DISTINCT FROM \'reserves\'')
        if project_ids:
            where.append('CAST("project_id" AS TEXT) = ANY(:ids)')
            params['ids'] = list(project_ids)
        if not where:
            return pd.read_sql(text(base), engine)
        try:
            return pd.read_sql(text(base + " WHERE " + " AND ".join(where)), engine, params=params)
        except Exception as e:
            print(f"  Filtered load failed ({type(e).__name__}); falling back to an unfiltered load.")
            return pd.read_sql(text(base), engine)

    print("Loading tables...")
    plant = get_sql_table("plant")
    country = get_sql_table("country")
    goget_project = get_sql_table("goget_project")

    plant = plant.rename(columns={'id': 'plant_table_id'})
    plant['plant_table_id'] = plant['plant_table_id'].astype(str)
    goget_project['project_id'] = goget_project['project_id'].astype(str)

    goget_project = goget_project.merge(plant[['plant_table_id', 'country_id', 'latitude', 'longitude']], left_on='project_id', right_on='plant_table_id', how='left')
    goget_project['country_id'] = pd.to_numeric(goget_project['country_id'], errors='coerce')
    goget_project['country_id'] = goget_project['country_id'].fillna(-1).astype('int64')
    goget_project = goget_project.merge(country[['id', 'gemName']], left_on='country_id', right_on='id', how='left')
    goget_project = goget_project.rename(columns={'gemName': 'Country/Area'})

    if selected_countries:
        goget_project = goget_project[goget_project['Country/Area'].isin(selected_countries)]

    goget_project = goget_project.drop(columns=['country_id'], errors='ignore')
    if 'id_y' in goget_project.columns: goget_project = goget_project.drop(columns=['id_y'])

    # reserves_production is by far the biggest table. Only load it if a check that uses it is selected.
    RP_STEPS = {"duplicates", "reserves", "production", "outliers"}
    reserves_production_merged = None
    if RP_STEPS & set(qc_steps):
        if {"reserves", "production"} & set(qc_steps):
            rp_cols = None   # these two write whole rows to CSV, so they need every column
        else:
            rp_cols = ['project_id', 'productionReserves', 'fuelDesc', 'quantity', 'quantityUnit', 'year']
            if "duplicates" in qc_steps:
                rp_cols.append('reserveClassification')
        rp_ids = goget_project['project_id'].unique().tolist() if selected_countries else None
        reserves_production = load_reserves_production(
            columns=rp_cols, project_ids=rp_ids,
            exclude_reserves=(set(qc_steps) & RP_STEPS == {"outliers"}))
        reserves_production['project_id'] = reserves_production['project_id'].astype(str)
        reserves_production_merged = reserves_production.merge(
            goget_project[["project_id", "namePrefix", "Country/Area"]],
            on="project_id", how="inner"
        )
        del reserves_production
        gc.collect()
        print(f"  reserves_production: {len(reserves_production_merged):,} rows loaded")
        _dup_ids = int(goget_project['project_id'].duplicated().sum())
        if _dup_ids:
            print(f"  WARNING: goget_project has {_dup_ids:,} repeated project_id rows. The join repeats "
                  f"production/reserves rows for them, which inflates row counts and memory.")

    if "wkt" in qc_steps:
        geo = get_sql_table("project_geospatial")
        geo['project_id'] = geo['project_id'].astype(str)
        goget_project_wkt = goget_project.merge(geo[['project_id', 'wkt']], on='project_id', how='left')
        goget_project_wkt_path = os.path.join(output_dir, f'goget_project_wkt_{timestamp}.xlsx')
        goget_project_wkt.to_excel(goget_project_wkt_path, index=False)
        print(f"WKT export saved to {goget_project_wkt_path}")

    # status_timeline is shared by the status export and the discovery-order check.
    # Load + filter it once (same logic as the export, so the country selection applies).
    status_timeline_merged = None
    if {"status", "discovery_order"} & set(qc_steps):
        status_timeline = get_sql_table("status_timeline")
        status_timeline['unit_id'] = status_timeline['unit_id'].astype(str)
        status_timeline_filtered = status_timeline.merge(goget_project[['project_id']], left_on='unit_id', right_on='project_id', how='inner')
        status_timeline_merged = status_timeline_filtered[status_timeline.columns].copy() # Explicitly create a copy
        for col in ['created', 'modified']:
            if col in status_timeline_merged.columns and pd.api.types.is_datetime64_any_dtype(status_timeline_merged[col]):
                if status_timeline_merged[col].dt.tz is not None:
                    status_timeline_merged[col] = status_timeline_merged[col].dt.tz_localize(None)
        if "status" in qc_steps:
            status_out = os.path.join(output_dir, f'status_timeline_check_{timestamp}.xlsx')
            status_timeline_merged.to_excel(status_out, index=False)
            print(f"Status timeline saved to {status_out}")

    if "discovery_order" in qc_steps:
        print("Discovery-year ordering check...")
        _st = status_timeline_merged
        _need = {'unit_id', 'status', 'substatus', 'year'}
        if not _need.issubset(_st.columns):
            print(f"  Skipped: status_timeline is missing column(s) {sorted(_need - set(_st.columns))}")
        else:
            _yr = pd.to_numeric(_st['year'], errors='coerce')
            _disc_units = set(_st.loc[_st['status'] == 'discovered', 'unit_id'])
            print(f"  {len(_st):,} status rows, {_st['unit_id'].nunique():,} units, {len(_disc_units):,} with a discovered entry")
            print(f"  Blind spot: {int(_yr.isnull().sum()):,} rows have a null year and cannot be compared "
                  f"({int((_yr.isnull() & _st['unit_id'].isin(_disc_units)).sum()):,} of them belong to units with a discovered entry)")

            first_prod = None
            if order_check_production:
                try:
                    _rp = load_reserves_production(
                        columns=['project_id', 'productionReserves', 'quantity', 'year'],
                        project_ids=(goget_project['project_id'].unique().tolist() if selected_countries else None))
                    _rp['project_id'] = _rp['project_id'].astype(str)
                    _rp = _rp[_rp['productionReserves'] == 'production']
                    _rp = _rp[pd.to_numeric(_rp['quantity'], errors='coerce') > 0]
                    _rp = _rp.assign(year=pd.to_numeric(_rp['year'], errors='coerce'))
                    first_prod = _rp.groupby('project_id')['year'].min().rename('first_production_year')
                    print(f"  Optional production comparison on: {len(first_prod):,} units have a production year > 0")
                except Exception as e:
                    print(f"  Production comparison skipped ({type(e).__name__}: {e})")

            order_viol = check_discovery_order(_st, first_production=first_prod)
            if order_viol.empty:
                print("  No discovery-year ordering violations found.")
            else:
                names = goget_project[[c for c in ['project_id', 'namePrefix', 'Country/Area'] if c in goget_project.columns]]
                names = names.drop_duplicates('project_id')
                order_viol = order_viol.merge(names, left_on='unit_id', right_on='project_id', how='left').drop(columns='project_id')
                lead = [c for c in ['unit_id', 'namePrefix', 'Country/Area', 'disc_year'] if c in order_viol.columns]
                order_viol = order_viol[lead + [c for c in order_viol.columns if c not in lead]]
                order_viol = order_viol.sort_values([c for c in ['Country/Area', 'unit_id'] if c in order_viol.columns]).reset_index(drop=True)
                order_viol['verified_year'] = ''
                order_viol['notes'] = ''
                order_out = os.path.join(output_dir, f'discovery_order_violations_{timestamp}.xlsx')
                order_viol.to_excel(order_out, index=False)
                print(f"  ERROR-LEVEL: {len(order_viol):,} unit(s) have a discovery year later than a FID/start-up/other milestone.")
                print(f"  Counts by comparison failed: " + "; ".join(
                    f"{c}: {int(order_viol['violated'].str.contains(c).sum()):,}"
                    for c in DISCOVERY_ORDER_COMPARISONS + ['first_production_year'] if c in order_viol.columns))
                print(f"  Review worksheet saved to {order_out}")

    if "duplicates" in qc_steps:
        def check_exact_duplicates(df):
            duplicates = df.groupby(['productionReserves', 'fuelDesc', 'reserveClassification', 'quantity', 'quantityUnit', 'year', 'project_id']).size().reset_index(name='counts')
            return duplicates[duplicates['counts'] > 1]
        dups = check_exact_duplicates(reserves_production_merged)
        if not dups.empty:
            dups.to_csv(os.path.join(output_dir, f"duplicate_full_entries_{timestamp}.csv"), index=False)
            print(f"Duplicate entries found and saved to duplicate_full_entries_{timestamp}.csv")
        else:
            print("No duplicate production entries found.")

    if "reserves" in qc_steps:
        reserves_df = reserves_production_merged[reserves_production_merged['productionReserves'].isin(['reserves'])].copy()
        reserves_df = reserves_df.sort_values(by=['project_id', 'fuelDesc', 'year'], ascending=[True, True, False])
        def reserves_classified(df):
            return df[df['reserveClassification'].isnull() | (df['reserveClassification'] == '')]
        unclassified_reserves = reserves_classified(reserves_df)
        if not unclassified_reserves.empty:
            unclassified_reserves.to_csv(os.path.join(output_dir, f"unclassified_reserves_entries_{timestamp}.csv"), index=False)
            print(f"Unclassified reserves entries found and saved.")
        else:
            print("No unclassified reserves entries found.")
        def reserve_not_timestamped(df):
            return df[df['year'].isnull() | (df['year'] == '')]
        timestamped_reserves = reserve_not_timestamped(reserves_df)
        if not timestamped_reserves.empty:
            timestamped_reserves.to_csv(os.path.join(output_dir, f"non_timestamped_reserves_entries_{timestamp}.csv"), index=False)
            print(f"Non-timestamped reserves entries found and saved.")
        else:
            print("No non-timestamped reserves entries found.")

    if "production" in qc_steps:
        production_df = reserves_production_merged[reserves_production_merged['productionReserves'].isin(['production', 'production design capacity'])].copy()
        production_df = production_df.sort_values(by=['project_id', 'fuelDesc', 'year'], ascending=[True, True, False])

        formatting_issues = []
        current_year = datetime.now().year

        required_fields = ['project_id', 'fuelDesc', 'year', 'quantity', 'quantityUnit', 'productionReserves']
        for field in required_fields:
            if field in production_df.columns:
                missing = production_df[production_df[field].isnull() | (production_df[field].astype(str).str.strip() == '')]
                if not missing.empty:
                    flagged = missing.copy()
                    flagged['formatting_issue'] = f'Missing required field: {field}'
                    formatting_issues.append(flagged)

        if 'year' in production_df.columns:
            # Filter out rows where 'year' is already considered missing (null or empty string)
            # This prevents 'Invalid year value' from being applied to genuinely missing years.
            non_missing_year_df = production_df[
                ~(production_df['year'].isnull() | (production_df['year'].astype(str).str.strip() == ''))
            ].copy()

            if not non_missing_year_df.empty:
                year_numeric = pd.to_numeric(non_missing_year_df['year'], errors='coerce')
                bad_year = non_missing_year_df[year_numeric.isnull() | (year_numeric < 1900) | (year_numeric > current_year + 10)].copy()
                if not bad_year.empty:
                    bad_year['formatting_issue'] = 'Invalid year value'
                    formatting_issues.append(bad_year)

        if 'quantity' in production_df.columns:
            qty_numeric = pd.to_numeric(production_df['quantity'], errors='coerce')
            bad_qty = production_df[qty_numeric.isnull() | (qty_numeric < 0)].copy()
            if not bad_qty.empty:
                bad_qty['formatting_issue'] = 'Quantity is missing or negative'
                formatting_issues.append(bad_qty)

        valid_prod_types = {'production', 'production design capacity'}
        if 'productionReserves' in production_df.columns:
            bad_type = production_df[~production_df['productionReserves'].str.strip().str.lower().isin(valid_prod_types)].copy()
            if not bad_type.empty:
                bad_type['formatting_issue'] = 'Unexpected productionReserves value'
                formatting_issues.append(bad_type)

        conversion_factors = get_sql_table("quantity_unit")
        unique_factors = conversion_factors.drop_duplicates(subset=['name'], keep='first')

        all_production_converted_df = production_df.copy()
        del production_df   # not used again below; frees a full-width copy
        all_production_converted_df['Fuel_quantity'] = all_production_converted_df['fuelDesc'] + ' ' + all_production_converted_df['quantityUnit']
        all_production_converted_df = all_production_converted_df.merge(
            unique_factors[['name', 'conversion_factor', 'common_unit']],
            left_on='Fuel_quantity', right_on='name', how='left'
        )
        all_production_converted_df = all_production_converted_df.rename(columns={
            'conversion_factor': 'Conversion factor',
            'common_unit': 'Common units',
        })
        all_production_converted_df['quantity_converted'] = all_production_converted_df['quantity'] * all_production_converted_df['Conversion factor']
        all_production_converted_df.loc[all_production_converted_df['Common units'] == 'million m³/y', 'shared unit (million boe/y)'] = all_production_converted_df['quantity_converted'] * .00588
        all_production_converted_df.loc[all_production_converted_df['Common units'].isin(['million bbl/y', 'shared unit (million boe/y)']), 'shared unit (million boe/y)'] = all_production_converted_df['quantity_converted']
        all_production_path = os.path.join(output_dir, f'all_production_converted_{timestamp}.csv')
        all_production_converted_df.to_csv(all_production_path, index=False)
        print(f"All production entries converted and saved to all_production_converted_{timestamp}.csv")

        valid_units = {'million m³/y', 'million bbl/y', 'million boe/y'}
        if 'Common units' in all_production_converted_df.columns:
            bad_units = all_production_converted_df[
                ~all_production_converted_df['Common units'].isin(valid_units)
            ][['project_id', 'namePrefix', 'Country/Area', 'fuelDesc', 'quantityUnit', 'Common units', 'productionReserves']].copy()
            if not bad_units.empty:
                bad_units['formatting_issue'] = 'Common units not in valid set'
                formatting_issues.append(bad_units)

        if formatting_issues:
            all_issues = pd.concat(formatting_issues, ignore_index=True)
            all_issues = all_issues.drop_duplicates(subset=['project_id', 'fuelDesc', 'year', 'quantity', 'formatting_issue'])
            issues_path = os.path.join(output_dir, f'production_formatting_issues_{timestamp}.csv')
            all_issues.to_csv(issues_path, index=False)
            print(f"Production formatting issues found ({len(all_issues)} rows). Saved to production_formatting_issues_{timestamp}.csv")
        else:
            print("No production formatting issues found.")

        all_production_converted_df = all_production_converted_df[all_production_converted_df['productionReserves'] == 'production']
        most_recent_production = all_production_converted_df.sort_values(by=['project_id', 'fuelDesc', 'year'], ascending=[True, True, False]).drop_duplicates(subset=['project_id', 'fuelDesc'], keep='first')

        gas_production_boe = most_recent_production[most_recent_production['Common units'] == 'million m³/y']['shared unit (million boe/y)'].sum()
        oil_production_boe = most_recent_production[most_recent_production['Common units'] == 'million bbl/y']['shared unit (million boe/y)'].sum()

        print(f"Total country production by fuel type:")
        print(f"  Gas: {gas_production_boe:,.2f} million boe/y ")
        print(f"  Oil: {oil_production_boe:,.2f} million boe/y")
        print(f"  Total: {most_recent_production['shared unit (million boe/y)'].sum():,.2f} million boe/y")
        del all_production_converted_df, most_recent_production, formatting_issues
        gc.collect()

    if "outliers" in qc_steps:
        print("Running outlier detection...")
        RATIO_THRESHOLD = outlier_ratio_threshold

        # Keep only the columns the check uses, and store each repeated string once (category)
        # instead of once per row. Rows are already limited to the selected countries by the join above.
        keep_cols = ['project_id', 'namePrefix', 'Country/Area', 'year', 'fuelDesc', 'quantityUnit', 'quantity']
        outlier_prod = reserves_production_merged.loc[
            reserves_production_merged['productionReserves'] != 'reserves', keep_cols].copy()
        del reserves_production_merged
        gc.collect()
        for c in ['project_id', 'namePrefix', 'Country/Area', 'fuelDesc', 'quantityUnit']:
            outlier_prod[c] = outlier_prod[c].astype('category')

        # Unit conversion: build "<fuel> <unit>" once per distinct fuel/unit pair (a handful),
        # not once per row (millions).
        fq = outlier_prod[['fuelDesc', 'quantityUnit']].drop_duplicates()
        fq['Fuel_quantity'] = fq['fuelDesc'].astype(object) + ' ' + fq['quantityUnit'].astype(object)

        conversion_factors_ol = get_sql_table("quantity_unit")
        unique_factors_ol = conversion_factors_ol.drop_duplicates(subset=['name'], keep='first')

        missing_fq = set(fq['Fuel_quantity'].unique()) - set(unique_factors_ol['name'].values)
        if missing_fq:
            print(f"  Warning: {len(missing_fq)} Fuel_quantity combos missing from conversion table: {missing_fq}")

        fq = fq.merge(
            unique_factors_ol[['name', 'conversion_factor', 'common_unit']],
            left_on='Fuel_quantity', right_on='name', how='left'
        )[['fuelDesc', 'quantityUnit', 'conversion_factor', 'common_unit']]
        outlier_prod = outlier_prod.merge(fq, on=['fuelDesc', 'quantityUnit'], how='left')
        outlier_prod['common_unit'] = outlier_prod['common_unit'].astype('category')
        outlier_prod['quantity_converted'] = outlier_prod['quantity'] * outlier_prod['conversion_factor']
        outlier_prod = outlier_prod.drop(columns=['quantity', 'conversion_factor'])

        print(f"  {outlier_prod['Country/Area'].notna().sum():,} of {len(outlier_prod):,} rows matched a country.")

        # Group = one project in one common unit. One groupby object, reused for both statistics.
        grp = outlier_prod.groupby(['project_id', 'common_unit'], observed=True, sort=False)['quantity_converted']
        group_median = grp.transform('median')
        group_size = grp.transform('count')
        del grp

        eligible = (group_size >= 2) & (group_median != 0)
        ratio = outlier_prod['quantity_converted'] / group_median

        too_high = eligible & (ratio > RATIO_THRESHOLD)
        too_low  = eligible & (ratio < 1 / RATIO_THRESHOLD)
        flagged = too_high | too_low

        print(f"  Total rows: {len(outlier_prod):,}  |  Outliers flagged: {int(flagged.sum()):,}")

        if flagged.any():
            # Only the flagged rows (a few percent) get text built for them.
            o = outlier_prod.loc[flagged, ['project_id', 'namePrefix', 'Country/Area', 'year', 'fuelDesc',
                                           'quantity_converted', 'common_unit']].copy()
            o['Project ID'] = 'L' + o['project_id'].astype(str)
            r, med = ratio[flagged], group_median[flagged]
            hi, lo = too_high[flagged], too_low[flagged]
            val = o['quantity_converted'].map('{:,.2f}'.format)
            o['outlier_reason'] = ''
            o.loc[hi, 'outlier_reason'] = (
                'Value (' + val[hi] + ') is ' + r[hi].map('{:.1f}'.format)
                + 'x the group median (' + med[hi].map('{:,.2f}'.format)
                + f') — exceeds {RATIO_THRESHOLD}x threshold'
            )
            o.loc[lo, 'outlier_reason'] = (
                'Value (' + val[lo] + ') is ' + r[lo].map('{:.6f}'.format)
                + 'x the group median (' + med[lo].map('{:,.2f}'.format)
                + f') — below 1/{RATIO_THRESHOLD}x threshold'
            )
            out_cols = ['Project ID', 'namePrefix', 'Country/Area', 'year',
                        'fuelDesc', 'quantity_converted', 'common_unit', 'outlier_reason']
            out_df = o[out_cols].astype({c: object for c in ['namePrefix', 'Country/Area', 'fuelDesc', 'common_unit']}).rename(columns={
                'namePrefix': 'Project Name',
                'year': 'Year',
                'fuelDesc': 'Fuel Description',
                'quantity_converted': 'Quantity (converted)',
                'common_unit': 'Quantity Unit (converted)',
            }).sort_values(['Project ID', 'Quantity Unit (converted)', 'Year'])
            outliers_path = os.path.join(output_dir, f'production_outliers_{timestamp}.csv')
            out_df.to_csv(outliers_path, index=False)
            print(f"  Outliers saved to production_outliers_{timestamp}.csv")
        else:
            print("  No outliers found.")
        del outlier_prod, group_median, group_size, ratio, eligible, too_high, too_low, flagged
        gc.collect()

    if "coordinates" in qc_steps:
        print("Running coordinate check (land + EEZ)...")
        if not GEO_AVAILABLE:
            print("  SKIPPED: geopandas/shapely unavailable. Re-run the install cell.")
        elif not geojson_path or not os.path.exists(geojson_path):
            print(f"  SKIPPED: land basemap not found at {geojson_path}")
        else:
            eez_use = None
            if eez_path and os.path.exists(eez_path):
                eez_use = _ensure_eez_gpkg(eez_path)
            else:
                print("  WARNING: no EEZ layer loaded. Offshore fields will be judged against")
                print("           land polygons only, which is the behaviour this check replaced.")

            coord_df = goget_project[['project_id', 'namePrefix', 'Country/Area',
                                      'latitude', 'longitude']].copy()
            coord_df['latitude'] = pd.to_numeric(coord_df['latitude'], errors='coerce')
            coord_df['longitude'] = pd.to_numeric(coord_df['longitude'], errors='coerce')

            # --- structural problems that are not really "wrong country" ---
            missing_coords = coord_df[coord_df['latitude'].isna() | coord_df['longitude'].isna()]
            out_of_range = coord_df[
                coord_df['latitude'].between(-90, 90, inclusive='both').eq(False)
                | coord_df['longitude'].between(-180, 180, inclusive='both').eq(False)
            ]
            null_island = coord_df[(coord_df['latitude'].abs() < 0.001)
                                   & (coord_df['longitude'].abs() < 0.001)]
            no_country = coord_df[coord_df['Country/Area'].isna()
                                  | (coord_df['Country/Area'].astype(str).str.strip() == '')]

            structural = []
            for frame, label in [(missing_coords, 'Missing latitude or longitude'),
                                 (out_of_range, 'Coordinate out of valid range'),
                                 (null_island, 'Coordinates are 0,0'),
                                 (no_country, 'No country assigned in database')]:
                if not frame.empty:
                    f = frame.copy()
                    f['issue'] = label
                    structural.append(f)
            if structural:
                struct_df = pd.concat(structural, ignore_index=True)
                struct_path = os.path.join(output_dir, f'coordinate_structural_issues_{timestamp}.csv')
                struct_df.to_csv(struct_path, index=False)
                print(f"  {len(struct_df)} structural coordinate issues -> "
                      f"coordinate_structural_issues_{timestamp}.csv")

            coord_valid = coord_df.dropna(subset=['latitude', 'longitude'])
            coord_valid = coord_valid[coord_valid['latitude'].between(-90, 90)
                                      & coord_valid['longitude'].between(-180, 180)].copy()

            if coord_valid.empty:
                print("  No valid coordinates to check.")
            else:
                print(f"  Resolving {len(coord_valid):,} points...")
                # One country at a time: only that country's region of the EEZ is in memory,
                # and it is released before the next country.
                parts = []
                for claim, grp in coord_valid.groupby('Country/Area', dropna=False, sort=False):
                    resolver = CountryResolver(geojson_path, eez_use,
                                               points=grp[['longitude', 'latitude']].to_numpy(),
                                               claim=claim)
                    parts.append(grp.apply(
                        lambda r: resolver.resolve(r['latitude'], r['longitude'],
                                                   r['Country/Area'], border_tol_km=border_tol_km),
                        axis=1, result_type='expand'))
                    del resolver
                    gc.collect()
                res = pd.concat(parts).reindex(coord_valid.index)
                coord_valid = pd.concat([coord_valid, res], axis=1)

                if flag_expected_cross_border:
                    known = coord_valid['namePrefix'].isin(EXPECTED_CROSS_BORDER)
                    coord_valid.loc[known & coord_valid['verdict'].eq('MISMATCH'),
                                    'verdict'] = 'expected-cross-border'

                full_path = os.path.join(output_dir, f'coordinate_check_full_{timestamp}.csv')
                coord_valid.to_csv(full_path, index=False)

                order = ['MISMATCH', 'unresolved-claim', 'disputed-boundary',
                         'borderline', 'expected-cross-border', 'OK']
                counts = coord_valid['verdict'].value_counts()
                print("  " + "  |  ".join(f"{v}: {counts.get(v, 0):,}" for v in order))

                review = coord_valid[coord_valid['verdict'] != 'OK'].copy()
                if not review.empty:
                    review['_rank'] = review['verdict'].map({v: i for i, v in enumerate(order)})
                    review = review.sort_values(
                        ['_rank', 'dist_to_claim_km'], ascending=[True, False]
                    ).drop(columns='_rank')
                    review_path = os.path.join(output_dir,
                                               f'mismatched_coordinates_{timestamp}.csv')
                    review.to_csv(review_path, index=False)
                    print(f"  Review queue ({len(review):,} rows) -> "
                          f"mismatched_coordinates_{timestamp}.csv")
                else:
                    print("  No coordinate mismatches found.")

    return f"QC process completed successfully. Output saved to: {output_dir}"

Mounted at /content/drive


In [ ]:
import psutil

def print_ram_usage(label=""):
    """Helper to print current system RAM usage."""
    process = psutil.Process(os.getpid())
    ram_mb = process.memory_info().rss / (1024 * 1024)
    virtual_mem = psutil.virtual_memory()
    print(f"[RAM] {label}: {ram_mb:,.1f} MB (System total: {virtual_mem.percent}%) ")

# Define hidden database URL, Conversions and outputs

In [ ]:
# grab hidden database URL from OnePassword as "GEM Database URL" - enter in the secrets panel
DATABASE_URL = userdata.get('DATABASE_URL')


if DATABASE_URL.startswith('postgres:'):
    DATABASE_URL = DATABASE_URL.replace('postgres:', 'postgresql:')

# name export path
QC_folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/QC/'
export_gen_path = "/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Alternative Formats of GOGET/"
export_qc_path = QC_folder + 'QC Output Dump/'

# Land basemap: Natural Earth 50m admin-0 countries
geojson_file = QC_folder + 'Ne-50M-Admin-0-Countries.geojson'

# Maritime boundaries: Marine Regions World EEZ.
# Set to None to fall back to land-only checking (not recommended - it is what
# mis-assigned every offshore field to the nearest coastline).
eez_file = QC_folder + 'World EEZ V12 20231025 polygons.geojson'

# Confirm the EEZ file is where we think it is
if eez_file and not os.path.exists(eez_file):
    print(f"EEZ file not found: {eez_file}")
    import glob
    found = (glob.glob(QC_folder + '*.geojson')
             + glob.glob(QC_folder + '*.gpkg')
             + glob.glob(QC_folder + '*.shp'))
    print('Geospatial files present in the QC folder:')
    for f in found:
        print('   ', os.path.basename(f))
else:
    print(f"EEZ file OK: {os.path.basename(eez_file) if eez_file else 'disabled'}")


EEZ file OK: World EEZ V12 20231025 polygons.geojson


QC Check descriptions
*   **Duplicates Check:** Identifies and reports exact duplicate entries in the production and reserves dataset based on key identifiers.
*   **Production Check:** Validates production data for correct formatting, data types, units, and calculates total production figures.
*   **Reserves Check:** Flags reserve entries that are missing classification details or a recorded year.
*   **Status Export:** Extracts and formats the status timeline data for projects, saving it to an Excel file.
*   **Discovery Order Check:** Error-level flag for units whose earliest `discovered` year is *strictly later* than their first actual-FID, in-development, operating (non-planned or any) or other non-exploration status year. Catches "discovered" rows that are really a source's report year. Exploration is excluded (it can precede discovery) and same-year is not a violation. Reports only (never edits years) as a review worksheet with blank `verified_year` / `notes` columns. Cannot detect fields with no later milestone, and rows with a null year cannot be compared (count is logged). Optional: also compare against the first year with production > 0.
*   **Coordinates Check:** Verifies that project coordinates fall inside the stated country. Onshore points are tested against Natural Earth 50m land polygons; offshore points are tested against Marine Regions EEZ polygons, so a North Sea field is judged on maritime jurisdiction rather than on whichever coastline happens to be nearest. Overlapping-claim and joint-regime waters pass for any claimant. Results are graded: `MISMATCH` (outside the claim by more than the border tolerance), `borderline` (within tolerance, usually basemap generalisation), `expected-cross-border` (known joint-development or unitised fields), `unresolved-claim` (country name absent from both layers). Structural problems (missing, zero, or out-of-range coordinates; no country assigned) are written to a separate file.
*   **WKT Export:** Extracts and exports Well-Known Text (WKT) geospatial data for each project to an Excel file.
*   **Outliers Check:** Detects unusually high or low production quantity values compared to typical values for similar projects and units.

In [ ]:
SELECTED_COUNTRIES = "" # @param {type:"string"}
DUPLICATES_CHECK = False # @param {type:"boolean"}
PRODUCTION_DATA = False # @param {type:"boolean"}
STATUS_TIMELINES = False # @param {type:"boolean"}
DISCOVERY_ORDER_CHECK = True # @param {type:"boolean"}
ORDER_CHECK_INCLUDE_PRODUCTION = False # @param {type:"boolean"}
COORDINATE_CHECK = False # @param {type:"boolean"}
RESERVES_CHECK = False # @param {type:"boolean"}
OUTLIER_CHECK = False # @param {type:"boolean"}
OUTLIER_RATIO_THRESHOLD = 10 # @param {type:"number"}
BORDER_TOLERANCE_KM = 5 # @param {type:"number"}
FLAG_EXPECTED_CROSS_BORDER = False # @param {type:"boolean"}
WKT_CHECK = False # @param {type:"boolean"}

# Ensure pandas, sqlalchemy, and text are available in this cell
import pandas as pd
from sqlalchemy import create_engine, text

# Check if DATABASE_URL is defined before using it
if 'DATABASE_URL' not in globals():
    DATABASE_URL = None # Assign None if not defined, to prevent NameError

if not DATABASE_URL:
    print("Please enter a DATABASE_URL.")
elif not geojson_file:
    print("Please specify a geojson_file path.")
elif not os.path.exists(geojson_file):
    print(f"Please upload '{geojson_file}' (needed for coordinate check). Or correct the file path, or unselect coordinate check.")
elif COORDINATE_CHECK and eez_file and not os.path.exists(eez_file):
    print(f"EEZ file '{eez_file}' not found. Fix the path in the cell above, or set")
    print("eez_file = None to run the coordinate check against land polygons only.")
else:
    qc_steps = []
    if DUPLICATES_CHECK: qc_steps.append("duplicates")
    if PRODUCTION_DATA: qc_steps.append("production")
    if STATUS_TIMELINES: qc_steps.append("status")
    if DISCOVERY_ORDER_CHECK: qc_steps.append("discovery_order")
    if COORDINATE_CHECK: qc_steps.append("coordinates")
    if RESERVES_CHECK: qc_steps.append("reserves")
    if OUTLIER_CHECK: qc_steps.append("outliers")
    if WKT_CHECK: qc_steps.append("wkt")

    countries = [c.strip() for c in SELECTED_COUNTRIES.split(",") if c.strip()]

    proceed_with_qc = True
    if countries: # Only check if specific countries were selected by the user
        try:
            temp_engine = create_engine(DATABASE_URL)
            with temp_engine.connect() as connection:
                db_countries_df = pd.read_sql(text('SELECT "gemName" FROM country'), connection)
            valid_db_country_names = set(db_countries_df['gemName'].dropna().str.strip().str.lower())

            user_selected_lower = {c.lower() for c in countries}
            if not any(user_country in valid_db_country_names for user_country in user_selected_lower):
                print("Country selected not in database")
                proceed_with_qc = False
        except Exception as e:
            print(f"Could not verify country existence due to database error: {e}")
            proceed_with_qc = False

    if proceed_with_qc:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = os.path.join(export_qc_path, f"qc_outputs_{SELECTED_COUNTRIES}_{run_id}")

        try:
            result = run_qc_checks(
                database_url=DATABASE_URL,
                output_dir=output_dir,
                selected_countries=countries if countries else None,
                qc_steps=qc_steps,
                geojson_path=geojson_file,
                eez_path=eez_file,
                border_tol_km=BORDER_TOLERANCE_KM,
                flag_expected_cross_border=FLAG_EXPECTED_CROSS_BORDER,
                outlier_ratio_threshold=OUTLIER_RATIO_THRESHOLD,
                order_check_production=ORDER_CHECK_INCLUDE_PRODUCTION
            )
            print(result)

            # Zip results
            zip_name = os.path.join(export_qc_path, f"qc_results_{SELECTED_COUNTRIES}_{run_id}.zip")
            with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for root, dirs, files_in_dir in os.walk(output_dir):
                    for file in files_in_dir:
                        file_path = os.path.join(root, file)
                        zipf.write(file_path, os.path.relpath(file_path, output_dir))

            print(f"Results zipped to {zip_name}.")

        except Exception as e:
            print(f"An error occurred: {e}")

Released cached land/EEZ layers to free RAM.
Loading tables...
Discovery-year ordering check...
  32,697 status rows, 14,581 units, 9,242 with a discovered entry
  Blind spot: 112 rows have a null year and cannot be compared (3 of them belong to units with a discovered entry)
  ERROR-LEVEL: 1 unit(s) have a discovery year later than a FID/start-up/other milestone.
  Counts by comparison failed: first_actual_fid: 0; first_in_dev: 0; first_startup_nonplanned: 1; first_operating_any: 1; first_nondiscovered_row: 1
  Review worksheet saved to /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/QC/QC Output Dump/qc_outputs__20261105_230038/discovery_order_violations_2026-11-05_23-00-38.xlsx
QC process completed successfully. Output saved to: /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/QC/QC Output Dump/qc_outputs__20261105_230038
Results zipped to /conte